# Parte 5 — Bonus: LoRA sobre BERTweet
### Workshop: Clasificación de Emociones en Twitter

BERTweet tiene ~110M parámetros — el doble que DistilBERT. LoRA permite adaptarlo con una fracción mínima de los parámetros.

BERTweet usa la arquitectura de BERT, cuyas proyecciones de atención tienen nombres distintos a DistilBERT:

| Modelo | Proyecciones |
|---|---|
| DistilBERT | `q_lin`, `k_lin`, `v_lin`, `out_lin` |
| BERTweet / BERT | `query`, `key`, `value`, `dense` |

**Prerequisito:** haber ejecutado `part-1-data.ipynb` y `part-2-pipeline.ipynb`

In [ ]:
# !pip install peft -q
%run part-2-pipeline.ipynb

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

MODEL_CHECKPOINT = "vinai/bertweet-base"
HF_REPO          = "your-username/tweeteval-emotion-bertweet-lora"  # <-- cambia esto
LR_LORA          = 2e-4   # más alto que full FT — pocos parámetros entrenables

## Dataset tokenizado

Reutilizamos el tokenizador de BERTweet del experimento anterior.

In [ ]:
from transformers import AutoTokenizer

tok_bertweet = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT, normalization=True)
ds_bertweet  = make_tokenized_dataset(tok_bertweet)
print(ds_bertweet)

## Inspección de módulos

Antes de aplicar LoRA, inspeccionamos los nombres de las capas lineales de BERTweet.

In [ ]:
from transformers import AutoModelForSequenceClassification

base_bt = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT, num_labels=NUM_LABELS,
    id2label=ID2LABEL, label2id=LABEL2ID)

print("Módulos lineales de BERTweet:")
for name, module in base_bt.named_modules():
    if isinstance(module, torch.nn.Linear):
        print(f"  {name}  [{module.in_features} → {module.out_features}]")

### 📝 TODO 5.1 — Configurar y aplicar LoRA a BERTweet

In [ ]:
# TODO 5.1 ── LoRA sobre BERTweet
# ─────────────────────────────────────────────────────────────────────────────
# 1. Define un LoraConfig con:
#    - task_type=TaskType.SEQ_CLS
#    - r=8, lora_alpha=16, lora_dropout=0.1
#    - target_modules: proyecciones query y value de BERTweet
#      (observa los nombres en la celda de inspección de arriba)
#    - modules_to_save: la cabeza clasificadora
#
# 2. Aplica LoRA con get_peft_model() e imprime los parámetros entrenables

# YOUR CODE HERE
# lora_config        = LoraConfig(...)
# model_bertweet_lora = get_peft_model(base_bt, lora_config)
# model_bertweet_lora.print_trainable_parameters()

### 📝 TODO 5.2 — Entrenar BERTweet-LoRA

In [ ]:
# TODO 5.2 ── Entrenar BERTweet-LoRA
# ─────────────────────────────────────────────────────────────────────────────
# Usa make_trainer() con lr=LR_LORA y output_dir="./checkpoints/bertweet-lora"
# Dataset: ds_bertweet (ya está tokenizado)
# Guarda los resultados en metrics_bertweet_lora y el trainer en trainer_lora

# YOUR CODE HERE

## Push to Hub

Para subir el modelo LoRA al Hub necesitamos fusionarlo primero (`merge_and_unload`), que aplica $W' = W + BA$ y devuelve un modelo estándar sin overhead PEFT.

In [ ]:
merged = model_bertweet_lora.merge_and_unload()
merged.push_to_hub(HF_REPO, commit_message="BERTweet LoRA r=8 — merged")
tok_bertweet.push_to_hub(HF_REPO)
print(f"Modelo publicado en: https://huggingface.co/{HF_REPO}")